# Daily morning board

Run top → bottom after the CLI morning loop (`grade_projections`, `log_projections`, `odds_board`, `poll_odds open`).
Pure slate/board generator — for CLV, PnL, edge-bin, and other results analysis,
see `production/notebooks/results_dashboard.ipynb` instead.

1. **Yesterday** — graded `expected_K` vs actuals  
2. **Today** — preferred SP projections  
3. **Edges** — model × live DK/FD (`recommendations.parquet`)

Batting orders = RotoGrinders; dual SP rows collapse to **preferred** (MLB on disagreement).

In [1]:
from __future__ import annotations

import pickle
import subprocess
import sys
import tempfile
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "Python").exists() and (candidate / "production").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(f"Cannot find src/Python from cwd={Path.cwd()}")

sys.path.insert(0, str(ROOT / "src"))

WORKER = ROOT / "production" / "notebooks" / "_notebook_score.py"
ALLOW_STALE = True
# Overnight RG often still shows yesterday after ET midnight. If the board
# scorer says cards match YESTERDAY, set: SLATE_DATE = YESTERDAY
# Leave None for "today ET" once RG has flipped.
SLATE_DATE: date | None = None
QUIET_WARNINGS = True
ET = ZoneInfo("America/New_York")
TODAY = datetime.now(ET).date()
YESTERDAY = TODAY - timedelta(days=1)

GRADED_PATH = ROOT / "artifacts" / "projection_log" / "graded.parquet"
REC_PATH = ROOT / "artifacts" / "odds_log" / "recommendations.parquet"
LEDGER_PATH = ROOT / "artifacts" / "odds_log" / "ledger.parquet"


def _attach_pitcher_team(df: pl.DataFrame) -> pl.DataFrame:
    if "is_home" not in df.columns or "away_team" not in df.columns:
        return df
    return df.with_columns(
        pl.when(pl.col("is_home"))
        .then(pl.col("home_team"))
        .otherwise(pl.col("away_team"))
        .alias("pitcher_team"),
    )


def show_scrollable(df: pl.DataFrame, height: int = 420):
    pdf = df.to_pandas()
    table = pdf.to_html(index=False, classes="proj-board", na_rep="—")
    display(
        HTML(
            f"""
<style>
  .proj-scroll {{ max-height: {height}px; overflow: auto; border: 1px solid #4443; border-radius: 6px; }}
  .proj-scroll table.proj-board {{ border-collapse: collapse; width: max-content; min-width: 100%; font-size: 13px; }}
  .proj-scroll thead th {{ position: sticky; top: 0; background: var(--jp-layout-color1, #1e1e1e); z-index: 1; text-align: left; padding: 6px 10px; white-space: nowrap; }}
  .proj-scroll tbody td {{ text-align: left; padding: 4px 10px; white-space: nowrap; }}
</style>
<div class="proj-scroll">{table}</div>
"""
        )
    )


def _summary(df: pl.DataFrame, label: str) -> dict:
    if df.is_empty():
        return {"label": label, "n": 0}
    ek = df["expected_K"].to_numpy()
    ak = df["actual_K"].to_numpy()
    resid = ek - ak
    return {
        "label": label,
        "n": int(df.height),
        "mae_K": round(float(np.mean(np.abs(resid))), 3),
        "bias_K": round(float(np.mean(resid)), 3),
        "rmse_K": round(float(np.sqrt(np.mean(resid**2))), 3),
        "corr": round(float(np.corrcoef(ek, ak)[0, 1]), 3) if len(ek) > 1 else None,
        "within_1K": round(float(np.mean(np.abs(resid) <= 1.0)), 3),
    }


print("repo:", ROOT)
print("today ET:", TODAY, "| yesterday ET:", YESTERDAY)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
today ET: 2026-08-06 | yesterday ET: 2026-08-05


## 1. Score today's slate

Needed for today's projections below. Quiet worker subprocess.

In [2]:
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as tmp:
    out_path = Path(tmp.name)

cmd = [sys.executable, str(WORKER), str(out_path)]
if ALLOW_STALE:
    cmd.append("--allow-stale")
if SLATE_DATE is not None:
    cmd.extend(["--date", SLATE_DATE.isoformat()])
if QUIET_WARNINGS:
    cmd.append("--quiet")

proc = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(
        "Board scorer failed.\n"
        f"cmd: {' '.join(cmd)}\nstdout:\n{proc.stdout}\nstderr:\n{proc.stderr}"
    )

payload = pickle.loads(out_path.read_bytes())
out_path.unlink(missing_ok=True)

board: pl.DataFrame = payload["board"]
preferred: pl.DataFrame = payload["preferred"]
build_meta = payload["build_meta"]
report = payload["report"]
slate_date = str(build_meta.get("slate_date"))

print(
    "slate:", slate_date,
    "| preferred:", preferred.height,
    "| mean xK:", round(float(report["mean_expected_K"]), 3),
)
print("rolling_max:", build_meta.get("rolling_max_date"), "stale_days:", build_meta.get("stale_days"))

slate: 2026-08-06 | preferred: 22 | mean xK: 4.846
rolling_max: 2026-08-05 stale_days: 1


## 2. Yesterday — projections vs actuals

From `graded.parquet`. Excludes pregame OOS and abbreviated outings when those flags exist.

`residual_K = expected_K − actual_K` (positive = over-projected).

In [3]:
matched = pl.DataFrame()
prev_day = pl.DataFrame()
if not GRADED_PATH.exists():
    print(f"Missing {GRADED_PATH}. Run grade_projections after Level 1 has actuals.")
else:
    graded = pl.read_parquet(GRADED_PATH).with_columns(pl.col("game_date").cast(pl.Date))
    if "is_preferred" in graded.columns:
        graded = graded.filter(pl.col("is_preferred"))
    if "grade_preferred_only" in graded.columns:
        pref_g = graded.filter(pl.col("grade_preferred_only") == True)  # noqa: E712
        if pref_g.height:
            graded = pref_g
    if "is_out_of_support" in graded.columns:
        graded = graded.filter(~pl.col("is_out_of_support").fill_null(False))
    if "is_abbreviated_outing" in graded.columns:
        graded = graded.filter(~pl.col("is_abbreviated_outing").fill_null(False))
    matched = graded.filter(pl.col("has_actual") & pl.col("actual_K").is_not_null())
    prev_day = matched.filter(pl.col("game_date") == YESTERDAY)

    display(pl.DataFrame([_summary(prev_day, f"yesterday_{YESTERDAY}")]).to_pandas())
    if prev_day.is_empty():
        print(f"No graded actuals for {YESTERDAY} yet.")
    else:
        prev = prev_day
        if "pitcher_team" not in prev.columns:
            prev = _attach_pitcher_team(prev)
        gcols = [
            c
            for c in (
                "player_name",
                "pitcher_team",
                "away_team",
                "home_team",
                "expected_K",
                "actual_K",
                "residual_K",
                "projected_tbf",
                "actual_PA",
            )
            if c in prev.columns
        ]
        prev = (
            prev.select(gcols)
            .with_columns(
                [
                    pl.col(c).round(2)
                    for c in ("expected_K", "residual_K", "projected_tbf")
                    if c in prev.columns
                ]
            )
            .sort("residual_K")
        )
        show_scrollable(prev, height=320)

,label,n,mae_K,bias_K,rmse_K,corr,within_1K
0,yesterday_2026-08-05,28,2.097,-0.896,2.644,0.512,0.321


player_name,pitcher_team,away_team,home_team,expected_K,actual_K,residual_K,projected_tbf,actual_PA
Tanner Bibee,CLE,NYM,CLE,4.72,10.0,-5.28,23.38,22.0
Kyle Harrison,MIL,PIT,MIL,4.84,10.0,-5.16,15.62,16.0
Mitch Bratt,AZ,SD,AZ,4.51,9.0,-4.49,21.17,24.0
Jacob Lopez,ATH,ATH,CIN,4.58,9.0,-4.42,21.21,19.0
Noah Cameron,KC,MIN,KC,4.59,9.0,-4.41,23.67,28.0
Trevor Rogers,BAL,LAA,BAL,5.21,9.0,-3.79,22.92,21.0
Eury Perez,MIA,MIA,ATL,5.62,9.0,-3.38,22.81,24.0
Hunter Brown,HOU,TOR,HOU,5.01,8.0,-2.99,23.44,23.0
Sonny Gray,BOS,CWS,BOS,5.25,8.0,-2.75,23.08,23.0
Christian Scott,NYM,NYM,CLE,4.79,7.0,-2.21,21.35,24.0


## 3. Today — preferred projections

`pitcher_team`, name, matchup, `xK`, and over-line probabilities.

In [4]:
pref = _attach_pitcher_team(preferred)
p_cols = [c for c in pref.columns if c.startswith("p_over_")]
front = [
    c
    for c in ("pitcher_team", "player_name", "away_team", "home_team", "expected_K")
    if c in pref.columns
]

round_exprs = [pl.col(c).round(3) for c in p_cols if pref[c].dtype in (pl.Float32, pl.Float64)]
if "expected_K" in pref.columns:
    round_exprs.append(pl.col("expected_K").round(2))

today_view = (
    pref.select(front + p_cols)
    .sort("expected_K", descending=True)
    .with_columns(round_exprs)
)

print(len(today_view))
show_scrollable(today_view, height=480)

22


pitcher_team,player_name,away_team,home_team,expected_K,p_over_2_5,p_over_2_5_cal,p_over_3_5,p_over_3_5_cal,p_over_4_5,p_over_4_5_cal,p_over_5_5,p_over_5_5_cal,p_over_6_5,p_over_6_5_cal,p_over_7_5,p_over_7_5_cal,p_over_8_5,p_over_8_5_cal,p_over_9_5,p_over_9_5_cal
PHI,Cristopher Sanchez,WSH,PHI,6.51,0.974,0.965,0.919,0.897,0.814,0.787,0.659,0.661,0.479,0.489,0.309,0.335,0.175,0.204,0.087,0.110
TOR,Dylan Cease,TOR,CHC,6.45,0.971,0.962,0.911,0.888,0.799,0.772,0.639,0.642,0.458,0.469,0.290,0.317,0.161,0.190,0.078,0.100
SEA,Bryce Miller,DET,SEA,6.37,0.972,0.963,0.913,0.890,0.802,0.774,0.640,0.643,0.456,0.467,0.286,0.313,0.156,0.185,0.074,0.096
PIT,Braxton Ashcraft,PIT,MIL,6.34,0.969,0.959,0.905,0.880,0.786,0.758,0.620,0.622,0.434,0.447,0.267,0.295,0.143,0.171,0.066,0.087
MIL,Dustin May,PIT,MIL,6.05,0.965,0.955,0.896,0.870,0.771,0.742,0.597,0.600,0.409,0.423,0.245,0.274,0.127,0.154,0.057,0.075
BAL,Brandon Young,LAA,BAL,5.41,0.930,0.910,0.819,0.781,0.652,0.622,0.458,0.463,0.281,0.300,0.149,0.178,0.069,0.089,0.027,0.039
NYM,Nolan McLean,NYM,CLE,5.36,0.931,0.912,0.821,0.784,0.654,0.625,0.461,0.466,0.284,0.302,0.151,0.180,0.070,0.091,0.028,0.040
BOS,Ranger Suarez,CWS,BOS,5.20,0.915,0.892,0.788,0.747,0.606,0.577,0.407,0.412,0.236,0.255,0.117,0.144,0.050,0.067,0.018,0.027
ATH,Mason Barnett,ATH,CIN,4.85,0.897,0.871,0.756,0.711,0.563,0.535,0.364,0.369,0.202,0.221,0.096,0.120,0.039,0.054,0.013,0.021
CIN,Andrew Abbott,ATH,CIN,4.73,0.871,0.841,0.711,0.663,0.508,0.483,0.313,0.319,0.165,0.184,0.075,0.096,0.029,0.041,0.010,0.015


## 4. Today — edges (model × books)

Reads `artifacts/odds_log/recommendations.parquet` from `production/odds/odds_board.py`.
Shows **BET** rows (≥8%, in-support) by default.

In [5]:
SHOW_ALL_EDGES = False  # True → include skip / OOS

if not REC_PATH.exists():
    print(f"Missing {REC_PATH}. Run: python production/odds/odds_board.py --unit 50")
else:
    rec = pl.read_parquet(REC_PATH)
    if "game_date" in rec.columns:
        rec = rec.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("game_date"))
        rec = rec.filter(pl.col("game_date") == slate_date[:10])
    if not SHOW_ALL_EDGES and "recommendation" in rec.columns:
        edges = rec.filter(pl.col("recommendation") == "BET")
    else:
        edges = rec
    edge_cols = [
        c
        for c in (
            "recommendation",
            "pitcher_team",
            "player_name",
            "away_team",
            "home_team",
            "expected_K",
            "book",
            "line",
            "best_side",
            "best_price",
            "edge",
            "units",
            "stake",
            "oos_reason",
        )
        if c in edges.columns
    ]
    view = edges.select(edge_cols)
    if "edge" in view.columns:
        view = view.with_columns((pl.col("edge") * 100).round(1).alias("edge_pct")).drop("edge")
    if "expected_K" in view.columns:
        view = view.with_columns(pl.col("expected_K").round(2))
    if view.is_empty():
        print("No BET rows for this slate (or empty recommendations).")
    else:
        show_scrollable(view, height=360)
    print(f"recommendations n={rec.height} shown={view.height if not view.is_empty() else 0}")

recommendation,pitcher_team,player_name,away_team,home_team,expected_K,book,line,best_side,best_price,units,stake,oos_reason,edge_pct
BET,TOR,Dylan Cease,TOR,CHC,6.45,draftkings,7.5,under,-111.0,1.31,65.58,None,18.7
BET,CLE,Foster Griffin,NYM,CLE,4.56,draftkings,5.5,under,-129.0,1.37,68.29,None,18.3
BET,DET,Framber Valdez,DET,SEA,4.24,fanduel,4.5,under,126.0,1.02,50.93,None,16.8
BET,BOS,Ranger Suarez,CWS,BOS,5.20,draftkings,5.5,under,120.0,0.97,48.46,None,15.9
BET,ATL,Martin Perez,MIA,ATL,3.56,draftkings,3.5,under,129.0,0.87,43.72,None,14.9
BET,SEA,Bryce Miller,DET,SEA,6.37,fanduel,5.5,over,-113.0,0.95,47.34,None,14.2


recommendations n=22 shown=6
